# RefusalBench baseline: Qwen1.5-7B-Chat (one click)

**To run everything:**
1. Runtime > Change runtime type > **T4 GPU** (once)
2. *(Only for the judge)* click the key icon on the left > Add new secret > name `GEMINI_API_KEY`, paste the key, switch "Notebook access" on. *If the GitHub repo is private, also add a secret `GITHUB_TOKEN`.*
3. **Runtime > Run all.** Keep this tab open and the computer awake.

If Colab disconnects, just click **Run all** again: it resumes where it stopped and never repeats finished work.
Results are saved on your Google Drive in `MyDrive/refusalbench-results/`.

In [ ]:
# 1) Setup: Drive, code, packages, secrets
import os
from google.colab import drive
drive.mount('/content/drive')

REPO = 'aidepartmentrah-pixel/refusalbench-reproduction'
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
url = f'https://{token + "@" if token else ""}github.com/{REPO}.git'

if not os.path.exists('/content/refusalbench-reproduction'):
    !git clone -q {url} /content/refusalbench-reproduction
os.chdir('/content/refusalbench-reproduction')
!git pull -q
!pip install -q -r requirements-colab.txt

try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('Gemini key found: the judge will run.')
except Exception:
    print('No GEMINI_API_KEY secret: generation will run, the judge will wait until you add it and click Run all again.')

In [ ]:
# 2) GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3) THE WHOLE BASELINE: 10 -> check -> 100 -> check -> all 1,600 -> judge -> metrics -> report
!python scripts/run_all.py --config Benchmark-Baseline-Reproduction/configs/qwen15_7b_baseline.yaml --out /content/drive/MyDrive/refusalbench-results

In [ ]:
# 4) Show the report, and offer a download copy (the Drive copy is already saved)
import os
d = '/content/drive/MyDrive/refusalbench-results/qwen15_7b_baseline'
print(open(f'{d}/reproduction_report.md').read()[:6000] if os.path.exists(f'{d}/reproduction_report.md') else 'no report yet')
try:
    from google.colab import files
    !cd /content/drive/MyDrive/refusalbench-results && zip -qr /content/qwen15_7b_baseline.zip qwen15_7b_baseline
    files.download('/content/qwen15_7b_baseline.zip')
except Exception as e:
    print('download skipped:', e)